# Enterprise RAG Pipeline Notebook

This notebook provides a professional, step-by-step workflow for the `enterprise_rag_pipeline.py` application.

It covers document ingestion from Azure Blob Storage or local files, chunking, embedding generation through Azure AI Foundry, Azure AI Search index creation, retrieval, grounded answer generation, and evaluation.

## End-to-End Flow

```mermaid
flowchart LR
    B[Azure Blob Storage or Local Docs] --> C[Chunk Documents]
    C --> E[Generate Embeddings in Foundry]
    E --> S[Azure AI Search Vector Index]
    Q[User Question] --> R[Hybrid Retrieval]
    S --> R
    R --> F[Metadata Filters]
    F --> G[Grounded Generation]
    G --> A[Answer with Citations]
    A --> V[Evaluation]
```

The notebook uses the same functions as the Python app, so changes in `enterprise_rag_pipeline.py` are automatically reflected here.

## 1. Install Required Libraries

Run this once in a fresh environment. If packages are already installed, this cell will simply confirm them.

In [11]:
# Brief logic: Install all packages required by the enterprise RAG application.
# The requirements file keeps notebook and application dependencies aligned.
%pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Load Configuration

This step loads `.env` values and displays only non-secret configuration. API keys are intentionally masked.

In [12]:
# Brief logic: Load environment variables and display the active enterprise RAG configuration safely.
# Secret values such as API keys are masked to avoid accidental exposure in notebook output.
import os
from dotenv import load_dotenv

load_dotenv()

config_keys = [
    "FOUNDRY_PROJECT_ENDPOINT",
    "AZURE_OPENAI_ENDPOINT",
    "MODEL_DEPLOYMENT_NAME",
    "EMBEDDING_DEPLOYMENT_NAME",
    "AI_SEARCH_SERVICE_ENDPOINT",
    "AI_SEARCH_INDEX_NAME",
    "RAG_SOURCE",
    "RAG_DOCS_DIR",
    "BLOB_CONTAINER_URL",
    "RAG_EVAL_FILE",
]

for key in config_keys:
    print(f"{key}: {os.getenv(key)}")

print("AI_SEARCH_API_KEY: ***masked***")

FOUNDRY_PROJECT_ENDPOINT: https://ajay-agent-project111-resource.services.ai.azure.com/api/projects/ajay-agent-project111
AZURE_OPENAI_ENDPOINT: https://ajay-agent-project111-resource.openai.azure.com/
MODEL_DEPLOYMENT_NAME: ajay-gpt-4o
EMBEDDING_DEPLOYMENT_NAME: text-embedding-3-small
AI_SEARCH_SERVICE_ENDPOINT: https://ajaysearchservice222.search.windows.net
AI_SEARCH_INDEX_NAME: ajay-enterprise-rag-demo11
RAG_SOURCE: local
RAG_DOCS_DIR: ./data/documents
BLOB_CONTAINER_URL: https://ajayaifoundrystoarg11.blob.core.windows.net/ajayragblob22
RAG_EVAL_FILE: ./eval/eval_questions.jsonl
AI_SEARCH_API_KEY: ***masked***


## 3. Import Pipeline Functions

The notebook imports the application functions instead of duplicating logic. This keeps the notebook clean and production-friendly.

In [13]:
# Brief logic: Import reusable functions from the application module.
# Reloading ensures the notebook picks up recent edits to enterprise_rag_pipeline.py.
import importlib
import enterprise_rag_pipeline

importlib.reload(enterprise_rag_pipeline)

from enterprise_rag_pipeline import (
    answer_question,
    create_or_update_index,
    embed_texts,
    evaluate,
    get_openai_client,
    ingest_documents,
    load_source_documents,
    require_env,
    retrieve,
)

## 4. Validate Required Settings

This cell checks that the required `.env` values are present before you call Azure services.

In [14]:
# Brief logic: Fail early if any required setting is missing or still contains a placeholder.
# This avoids confusing Azure SDK errors later in the workflow.
required_keys = [
    "FOUNDRY_PROJECT_ENDPOINT",
    "AZURE_OPENAI_ENDPOINT",
    "MODEL_DEPLOYMENT_NAME",
    "EMBEDDING_DEPLOYMENT_NAME",
    "AI_SEARCH_SERVICE_ENDPOINT",
    "AI_SEARCH_INDEX_NAME",
    "AI_SEARCH_API_KEY",
    "RAG_SOURCE",
    "RAG_EVAL_FILE",
]

if os.getenv("RAG_SOURCE", "local").lower() == "blob":
    required_keys.append("BLOB_CONTAINER_URL")
else:
    required_keys.append("RAG_DOCS_DIR")

for key in required_keys:
    require_env(key)

print("Configuration validation passed.")

Configuration validation passed.


## 5. Preview Source Documents

This reads documents from the active source. If `RAG_SOURCE="blob"`, your Azure login must have `Storage Blob Data Reader` access to the container.

In [15]:
# Brief logic: Load documents from Azure Blob Storage or the local documents folder.
# This preview confirms the source is reachable before indexing starts.
docs_preview = load_source_documents()

print(f"Loaded {len(docs_preview)} chunks from source: {os.getenv('RAG_SOURCE')}")

if docs_preview:
    sample = docs_preview[0]
    print("\nSample metadata:")
    print({key: sample[key] for key in ["title", "source_file", "department", "security_level", "chunk_id"]})
    print("\nSample content preview:")
    print(sample["content"][:700])

Loaded 2 chunks from source: local

Sample metadata:
{'title': 'abc-benefits-policy', 'source_file': 'abc-benefits-policy.txt', 'department': 'HR', 'security_level': 'internal', 'chunk_id': 0}

Sample content preview:
Title: ABC Corp Benefits Policy Department: HR Security: internal ABC Corp provides medical insurance, paid leave, learning reimbursement, and retirement contribution benefits to eligible employees. Benefit eligibility depends on employment status, location, and job type. Contractors may have limited benefits unless a contract explicitly states otherwise. Questions about individual eligibility should be answered only when the retrieved source contains the relevant employee or policy details.


## 6. Verify Foundry Resource Embedding Deployment

This checks that your Azure AI Foundry resource endpoint and embedding deployment are reachable.

In [16]:
# Brief logic: Create an Azure OpenAI client for the Foundry resource and generate a tiny test embedding.
# The returned vector dimension is used by Azure AI Search index creation.
openai_client = get_openai_client()
probe_vector = embed_texts(openai_client, ["enterprise rag embedding probe"])[0]

print(f"Embedding deployment: {os.getenv('EMBEDDING_DEPLOYMENT_NAME')}")
print(f"Embedding vector dimensions: {len(probe_vector)}")

2026-06-02 23:18:32,578 INFO No environment configuration found.
2026-06-02 23:18:32,579 INFO ManagedIdentityCredential will use IMDS
2026-06-02 23:18:32,741 INFO Request URL: 'http://169.254.169.254/metadata/identity/oauth2/token?api-version=2018-02-01&resource=REDACTED'
Request method: 'GET'
Request headers:
    'User-Agent': 'azsdk-python-identity/1.25.3 Python/3.11.9 (Windows-10-10.0.26200-SP0)'
No body was attached to the request
2026-06-02 23:18:34,416 INFO DefaultAzureCredential acquired a token from AzureCliCredential
2026-06-02 23:18:35,542 INFO HTTP Request: POST https://ajay-agent-project111-resource.openai.azure.com/openai/deployments/text-embedding-3-small/embeddings?api-version=2024-10-21 "HTTP/1.1 200 OK"


Embedding deployment: text-embedding-3-small
Embedding vector dimensions: 1536


## 7. Ingest Documents into Azure AI Search

This creates or updates the Azure AI Search index, generates embeddings, and uploads chunks.

Set `RUN_INGEST = True` when you are ready to write to the Azure AI Search index.

In [17]:
# Brief logic: Run the full ingestion pipeline only when explicitly enabled.
# This cell writes chunks and vectors into the configured Azure AI Search index.
RUN_INGEST = False

if RUN_INGEST:
    ingest_documents()
    print("Ingestion completed.")
else:
    print("Ingestion skipped. Set RUN_INGEST = True to index documents.")

Ingestion skipped. Set RUN_INGEST = True to index documents.


## 8. Test Retrieval

This runs hybrid retrieval against Azure AI Search and prints the top retrieved chunks.

In [18]:
# Brief logic: Retrieve the most relevant chunks for a test question.
# Optional metadata filtering is useful for department, security, tenant, or group-based access patterns.
test_question = "Does the ABC Corp dataset mention employee records?"
metadata_filter = "department eq 'HR' and security_level eq 'internal'"

retrieved_chunks = retrieve(test_question, filter_expression=metadata_filter, top=3)

for index, item in enumerate(retrieved_chunks, start=1):
    print(f"\nResult {index}")
    print(f"Source: {item['source_file']} | Chunk: {item['chunk_id']} | Department: {item['department']}")
    print(item["content"][:900])

2026-06-02 23:18:35,562 INFO No environment configuration found.
2026-06-02 23:18:35,563 INFO ManagedIdentityCredential will use IMDS
2026-06-02 23:18:35,719 INFO Request URL: 'http://169.254.169.254/metadata/identity/oauth2/token?api-version=2018-02-01&resource=REDACTED'
Request method: 'GET'
Request headers:
    'User-Agent': 'azsdk-python-identity/1.25.3 Python/3.11.9 (Windows-10-10.0.26200-SP0)'
No body was attached to the request
2026-06-02 23:18:37,517 INFO DefaultAzureCredential acquired a token from AzureCliCredential
2026-06-02 23:18:38,663 INFO HTTP Request: POST https://ajay-agent-project111-resource.openai.azure.com/openai/deployments/text-embedding-3-small/embeddings?api-version=2024-10-21 "HTTP/1.1 200 OK"
2026-06-02 23:18:38,664 INFO Request URL: 'https://ajaysearchservice222.search.windows.net/indexes('ajay-enterprise-rag-demo11')?api-version=2026-04-01'
Request method: 'GET'
Request headers:
    'Accept': 'application/json;odata.metadata=minimal'
    'x-ms-client-reque


Result 1
Source: HR_Records.txt | Chunk: 0 | Department: HR
EmployeeID EmployeeName Department Email PhoneNumber Designation JoiningDate ManagerName LeaveBalance PayrollStatus SalaryBand Location EmploymentType HRComments EMP101 Ramesh Kumar IT ramesh.kumar@company.com 9876543210 Software Engineer 15-01-2023 Ganesh Patil 12 Active L2 Bangalore Full-Time Working on AI automation project EMP102 Ganesh Patil HR ganesh.patil@company.com 9876543211 HR Manager 20-06-2021 Suresh Rao 18 Active L4 Hyderabad Full-Time Handles payroll approvals EMP103 Priya Sharma Finance priya.sharma@company.com 9876543212 Financial Analyst 10-03-2022 Kavita Nair 10 Active L3 Pune Full-Time Manages quarterly reports EMP104 Suresh Rao Operations suresh.rao@company.com 9876543213 Operations Lead 05-11-2020 Anand Mehta 20 Active L5 Chennai Full-Time Supervises regional operations EMP105 Kavita Nair Marketing kavita.nair@company.com 9876543214 Marketing Specialist 12-02-2024 N

Result 2
Source: HR_Policies.txt | Ch

## 9. Generate a Grounded Answer

This retrieves relevant chunks, sends them to the Foundry model deployment, and asks the model to answer only from retrieved context.

In [19]:
# Brief logic: Ask a grounded RAG question using the retrieval and generation pipeline.
# The answer prompt requires citations using source_file and chunk_id.
question = "Does the ABC Corp dataset mention whether employee records exist in HR-Employee_Records?"
filter_expression = "department eq 'HR' and security_level eq 'internal'"

answer = answer_question(question, filter_expression=filter_expression)
print(answer)

2026-06-02 23:18:39,469 INFO No environment configuration found.
2026-06-02 23:18:39,470 INFO ManagedIdentityCredential will use IMDS
2026-06-02 23:18:39,633 INFO No environment configuration found.
2026-06-02 23:18:39,634 INFO ManagedIdentityCredential will use IMDS
2026-06-02 23:18:39,799 INFO Request URL: 'http://169.254.169.254/metadata/identity/oauth2/token?api-version=2018-02-01&resource=REDACTED'
Request method: 'GET'
Request headers:
    'User-Agent': 'azsdk-python-identity/1.25.3 Python/3.11.9 (Windows-10-10.0.26200-SP0)'
No body was attached to the request
2026-06-02 23:18:41,454 INFO DefaultAzureCredential acquired a token from AzureCliCredential
2026-06-02 23:18:42,446 INFO HTTP Request: POST https://ajay-agent-project111-resource.openai.azure.com/openai/deployments/text-embedding-3-small/embeddings?api-version=2024-10-21 "HTTP/1.1 200 OK"
2026-06-02 23:18:42,448 INFO Request URL: 'https://ajaysearchservice222.search.windows.net/indexes('ajay-enterprise-rag-demo11')?api-ver

Yes, the ABC Corp dataset mentions employee records in HR-Employee_Records. The dataset provided in "HR_Records.txt chunk 0" contains detailed employee information, including EmployeeID, EmployeeName, Department, Email, PhoneNumber, Designation, JoiningDate, ManagerName, LeaveBalance, PayrollStatus, SalaryBand, Location, EmploymentType, and HRComments. This indicates that employee records exist in the HR system.

Cited Source: HR_Records.txt chunk 0


## 10. Run Evaluation

This uses `eval/eval_questions.jsonl` to perform a lightweight quality check. It verifies that expected keywords appear in generated answers.

Set `RUN_EVAL = True` when you are ready to call retrieval and generation for each evaluation question.

In [20]:
# Brief logic: Run the evaluation suite only when explicitly enabled.
# This calls Azure AI Search and the Foundry model deployment for every eval case.
RUN_EVAL = False

if RUN_EVAL:
    evaluate()
else:
    print("Evaluation skipped. Set RUN_EVAL = True to run eval questions.")

Evaluation skipped. Set RUN_EVAL = True to run eval questions.


## Operational Notes

- Use `RAG_SOURCE="blob"` for enterprise ingestion from Azure Blob Storage.
- Use `RAG_SOURCE="local"` for offline demos with `data/documents`.
- Keep Search admin keys out of committed code and rotate exposed keys.
- For production access control, add user or group metadata to each chunk and enforce filters server-side.
- For deeper quality tracking, move from keyword evals to Foundry evaluations for groundedness, relevance, and citation accuracy.